# Generate analysis dataframe for hits with good triggers

In [ ]:
import h5py as h5 

import matplotlib.pyplot as plt
from matplotlib import cm, colors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib import colors, ticker

import numpy as np
import os, sys
import traceback
import glob
import pandas as pd

import h5flow 
plt.style.use('../../utils/dune.mplstyle')
from sklearn.cluster import DBSCAN

# Path to repo root (two directories above notebook)
light_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(light_root)

from light.PMT_analysis_utils import * 

# Path to repo root (two directories above notebook)
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(repo_root)

from utils.backtracking import get_charge_event_hits, hit_backtracker, get_ancestry # Now this works
from utils.my_ev_display import event_display

In [ ]:
# Configure your analysis here 
# This debug mode will process only the 10% of the dataset , use this if looking at source data
DEBUG = False

#input_dataset = "/global/cfs/cdirs/dune/www/data/2x2/reflows_run2/v0/flow/ColdOperations/data/2025_Operations_Cold/source/AmBe_1112"
#input_dataset = "/pscratch/sd/d/dunepro/mkramer/output/Reflow_2x2_Run2_v0p2/flow/source_ambe_bin2/one_trig_32us_window"
#input_dataset = "/pscratch/sd/d/dunepro/mkramer/output/Reflow_2x2_Run2_v0p2/flow/source_ambe_bin0/mod2_pmt_trig"
input_dataset = "/pscratch/sd/d/dunepro/mkramer/output/Reflow_2x2_Run2_v0p2/flow/source_ambe_bin3/one_pmt_trig_no_source"

MULTI = False

out_clusters = "../output/nosource_randtrig_ambe_bin2_one_trig_32us_window_eps1size4.csv"

In [ ]:
def filter_hits(hits_set):
    # Remove hits with negative energy
    hits_set = hits_set[hits_set[:,3] > 0.]
    # Remove hits with any NaN field
    hits_set = hits_set[~np.isnan(hits_set).any(axis=1)]
    return hits_set

In [ ]:
def CL_AmBe_analysis(input_file,file_id,single=True,clust_eps=1,clust_min_samples=4):
    '''
    Produces AmBe analysis objects

    Column names:
    x, y, z, E, Q, light_id, cluster_label, file_id

    '''

    h5_file = h5flow.data.H5FlowDataManager(input_file,'r')
    g_triggers = None
    b_triggers = None

    # Initialize output dataset 
    out_dataset = {
        "clusters":None
    }

    if(single):
        g_triggers, b_triggers = classify_triggers_single(h5_file,debug=False)
    
    else: 
        first_trig = check_first_trig(h5_file,291)
        parity_str = None
        if(first_trig):
            parity_str = "odd"
        else:
            parity_str = "even"
        print(f"Parity of this file: {parity_str}")
        g_triggers, b_triggers = classify_triggers(h5_file,parity_str,debug=False)

    #print(f"List of good triggers: {g_triggers}")

    # Retrieve products linked to good triggers 
    light_events = h5_file['light/events',g_triggers]
    light_wvfms = h5_file['light/wvfm',g_triggers]
    charge_events = h5_file['light/events','charge/events',g_triggers]
    charge_hits = h5_file['light/events','charge/events','charge/calib_prompt_hits',g_triggers]

    # Charge
    non_zero_charge_hits = charge_hits[charge_events.data['nhit'][:,0] >= 1]
    non_zero_charge_events = charge_events[charge_events.data['nhit'][:,0] >= 1]
    non_zero_charge_light_ev = light_events[charge_events.data['nhit'][:,0] >= 1] 

    E_clusters = []
    n_cluster = [] 
    n_hits_clusters = [] 
    cluster_light_ev_id = [] 

    clusterized_hits = [] 

    db = DBSCAN(eps=clust_eps,min_samples=clust_min_samples)
    for icharge in range(len(non_zero_charge_events)):
    #for icharge in range(2):
        test_event_hits = non_zero_charge_hits[:,0][icharge][0:non_zero_charge_events.data['nhit'][:,0][icharge]]
        this_event_light_id = non_zero_charge_light_ev['id'][icharge]
        hits_stack_unfiltered = np.column_stack((test_event_hits.data['x'],test_event_hits.data['y'],test_event_hits.data['z'],test_event_hits.data['E'],test_event_hits.data['Q'],np.ones(len(test_event_hits))*this_event_light_id))
        hits_stack = filter_hits(hits_stack_unfiltered)
        if(len(hits_stack)==0):
            continue
        labels = db.fit_predict(hits_stack[:,:3])
        hits_stack_label = np.column_stack((hits_stack[:,0],hits_stack[:,1],hits_stack[:,2],hits_stack[:,3],hits_stack[:,4],np.ones(len(hits_stack))*int(this_event_light_id),labels,np.ones(len(hits_stack))*int(file_id)))
        clusterized_hits.extend(hits_stack_label.tolist())
        temp_n_clusters = 0

        for l in np.unique(labels):
            if(l!=-1):
                this_l_hits = hits_stack_label[hits_stack_label[:,6]==l]
                n_hits_clusters.append(len(this_l_hits))
                E_clusters.append(np.sum(this_l_hits[:,3]))
                cluster_light_ev_id.append([l,this_event_light_id])
                temp_n_clusters+=1
            else:
                continue
        n_cluster.append(temp_n_clusters)

    out_dataset["clusters"] = clusterized_hits


    return out_dataset

In [ ]:
def get_rand_trig(input_file,file_id,n_trigs,single=True,clust_eps=1,clust_min_samples=4):
    '''
    Produces AmBe analysis objects without light trigger selection

    Column names:
    x, y, z, E, Q, light_id, cluster_label, file_id
    '''

    h5_file = h5flow.data.H5FlowDataManager(input_file,'r')
    g_triggers = None
    b_triggers = None

    # Initialize output dataset
    out_dataset = {
        "clusters":None
    }

    # Randomly sample triggers
    if(single):
        g_triggers = np.random.choice(h5_file['light/events/data']['id'], size=n_trigs, replace=False)
    
    else: 
        first_trig = check_first_trig(h5_file,291)
        parity_str = None
        if(first_trig):
            parity_str = "odd"
        else:
            parity_str = "even"
        print(f"Parity of this file: {parity_str}")
        g_triggers = np.random.choice(h5_file['light/events/data']['id'], size=n_trigs, replace=False)

    #print(f"List of good triggers: {g_triggers}")

    # Retrieve products linked to good triggers 
    light_events = h5_file['light/events',g_triggers]
    light_wvfms = h5_file['light/wvfm',g_triggers]
    charge_events = h5_file['light/events','charge/events',g_triggers]
    charge_hits = h5_file['light/events','charge/events','charge/calib_prompt_hits',g_triggers]

    # Charge
    non_zero_charge_hits = charge_hits[charge_events.data['nhit'][:,0] >= 1]
    non_zero_charge_events = charge_events[charge_events.data['nhit'][:,0] >= 1]
    non_zero_charge_light_ev = light_events[charge_events.data['nhit'][:,0] >= 1] 

    E_clusters = []
    n_cluster = [] 
    n_hits_clusters = [] 
    cluster_light_ev_id = [] 

    clusterized_hits = [] 

    db = DBSCAN(eps=clust_eps,min_samples=clust_min_samples)
    for icharge in range(len(non_zero_charge_events)):
    #for icharge in range(2):
        test_event_hits = non_zero_charge_hits[:,0][icharge][0:non_zero_charge_events.data['nhit'][:,0][icharge]]
        this_event_light_id = non_zero_charge_light_ev['id'][icharge]
        hits_stack_unfiltered = np.column_stack((test_event_hits.data['x'],test_event_hits.data['y'],test_event_hits.data['z'],test_event_hits.data['E'],test_event_hits.data['Q'],np.ones(len(test_event_hits))*this_event_light_id))
        hits_stack = filter_hits(hits_stack_unfiltered)
        if(len(hits_stack)==0):
            continue
        labels = db.fit_predict(hits_stack[:,:3])
        hits_stack_label = np.column_stack((hits_stack[:,0],hits_stack[:,1],hits_stack[:,2],hits_stack[:,3],hits_stack[:,4],np.ones(len(hits_stack))*int(this_event_light_id),labels,np.ones(len(hits_stack))*int(file_id)))
        clusterized_hits.extend(hits_stack_label.tolist())
        temp_n_clusters = 0

        for l in np.unique(labels):
            if(l!=-1):
                this_l_hits = hits_stack_label[hits_stack_label[:,6]==l]
                n_hits_clusters.append(len(this_l_hits))
                E_clusters.append(np.sum(this_l_hits[:,3]))
                cluster_light_ev_id.append([l,this_event_light_id])
                temp_n_clusters+=1
            else:
                continue
        n_cluster.append(temp_n_clusters)

    out_dataset["clusters"] = clusterized_hits


    return out_dataset

In [ ]:
all_clusters = []

for file_count, ifile in enumerate(os.listdir(input_dataset)):

    # Run over 10% of the dataset
    if DEBUG and file_count >= int(len(os.listdir(input_dataset))*0.1):
        break

    this_file = os.path.join(input_dataset, ifile)
    print(this_file)

    try:
        '''
        temp_out = CL_AmBe_analysis(
            this_file,
            file_count,
            single=not MULTI
        )
        '''
        temp_out = get_rand_trig(
            this_file,
            file_count,
            1000,
            single=not MULTI
        )
        all_clusters.extend(temp_out["clusters"])

    except Exception as e:
        print(f"[WARNING] Failed processing {this_file}: {e}")
        continue

all_clusters_array = np.array(all_clusters)

In [ ]:
# Inspect dimensions of different files

for ifile in np.unique(all_clusters_array[:,6]):
    print(f"Dimensions of file {ifile} {all_clusters_array[all_clusters_array[:,6]==ifile].shape}")

In [ ]:
df = pd.DataFrame(all_clusters_array, columns=['x', 'y', 'z', 'E', 'Q', 'light_id', 'cluster_label', 'file_id'])

df["id"] = (
    df["file_id"].astype(str)
    + "::" + df["light_id"].astype(str)
    + "::" + df["cluster_label"].astype(str)
)

df.to_csv(out_clusters)

In [ ]:
df.head()

In [ ]:
# Visualize clustering results

# Energy histogram of individual hits

plt.figure(figsize=(12, 6))

plt.hist(df['E'], color='green', histtype='step', bins=np.arange(0,10,0.2), label='Source Data')

plt.xlabel('Energy of Individual Hits Generated by Neutron (MeV)')
plt.ylabel('Hit Count')
plt.grid(True)
#plt.yscale('log')
#plt.ylim((0,10))
plt.legend()
plt.show()

#### Subsection for Generating Analysis Dataframe for MC

In [ ]:
def filter_hits(hits_set):
    # Remove hits with negative energy
    hits_set = hits_set[hits_set[:,3].astype(float) > 0.]
    # Remove hits with any NaN field
    hits_set = hits_set[~np.isnan(hits_set[:,:5].astype(float)).any(axis=1)]
    return hits_set

In [ ]:
def get_MC(MC_file, clust_eps=1,clust_min_samples=4):
    '''
    Load and cluster MC data, export as df
    '''

    file_id = 0
    mc_hit_arr = np.empty(9)

    # Load MC
    for file in glob.iglob(MC_file):
        with h5.File(file, 'r') as f:
            traj = f['mc_truth/trajectories/data']
            seg = f['mc_truth/segments/data']
            for event_id in np.unique(traj['event_id']):
                my_entry = event_id-1
                ev_traj= traj[traj['event_id']==event_id]
                ev_seg= seg[seg['event_id']==event_id]
                prompt_hits_backtrack, prompt_hits = get_charge_event_hits(my_entry,f)
                new_hits = hit_backtracker(prompt_hits_backtrack, prompt_hits, ev_seg, ev_traj)

                # Exclude negative hits
                positive_hits = new_hits[new_hits['E']>=0]

                # Use parent neutron id as light trigger id since they should correspond with each pmt id from data
                # All cluster ids are -2 for now as placeholder
                # All ids are 'nan' for now as placeholder
                mc_hit_arr = np.vstack([mc_hit_arr, np.column_stack([positive_hits['x'], positive_hits['y'], positive_hits['z'], positive_hits['E'], positive_hits['Q'], positive_hits['parent_neutron_id'], -2*np.ones(len(positive_hits['x'])), file_id*np.ones(len(positive_hits['x'])), np.full(len(positive_hits['x']),fill_value='nan')])])
            file_id += 1

    # Filter hits
    mc_hit_arr = filter_hits(mc_hit_arr)

    mc_df = pd.DataFrame(mc_hit_arr, columns=['x', 'y', 'z', 'E', 'Q', 'light_id', 'cluster_label', 'file_id', 'id'])

    # Perform clustering
    db = DBSCAN(eps=clust_eps,min_samples=clust_min_samples)
    # Generate a list of unique light trig_ids
    trig_ids = mc_df['file_id'].astype(str)+ '::' + mc_df['light_id'].astype(str)
    mc_hit_trig_ids = mc_hit_arr[:,7].astype(str) + '::' + mc_hit_arr[:,5].astype(str)
    for i in np.unique(mc_hit_trig_ids):
        labels = db.fit_predict(mc_hit_arr[:,:3][trig_ids==i].astype(float))
        # Change cluster labels to results based on the overlay
        mc_df.loc[trig_ids==i,'cluster_label'] = labels
        # temporarily define the ids of clustered hits as something identifiable
        mc_df.loc[trig_ids==i,'id'] = 'clustered'

    final_mc_df = mc_df.drop(np.where(mc_df['id']!='clustered')[0])
    final_mc_df['id']=final_mc_df['file_id'].astype(str) +'::'+ final_mc_df['light_id'].astype(str) +'::'+ final_mc_df['cluster_label'].astype(str)
    print(f'Total number of neutrons in MC file: {len(np.unique(mc_hit_trig_ids))}')
    return final_mc_df

In [ ]:
# Generate analysis dataframe

MC_data = '/global/cfs/cdirs/dune/users/lmlepin/2x2_neutron_prod/AmBe_top_mod2_PROD_03-21/FLOW/*.hdf5'
#MC_data = '/global/cfs/cdirs/dune/users/lmlepin/2x2_neutron_prod/AmBe_top_mod2_PROD_03-21/FLOW/2x2_QGSP_BERT_HP_AmBe_1774119367_0_TIME_MOD.FLOW.hdf5'

#df_MC = get_MC(MC_data)
df_MC = get_MC(MC_data, clust_eps=1, clust_min_samples=3)

In [ ]:
df_MC

In [ ]:
# Export the MC data as df

#df_MC.to_csv("../output/MC_AmBe_Prod_03-21_eps1size4.csv")
df_MC.to_csv("../output/eps1_minsamples3/MC_AmBe_Prod_05-29_eps1size3.csv")